# 04. Learning Curve

**Paper section:** §4.3 How much segmented text is enough? (Figure 3).
**What it computes:** Trains the transitional-probability segmenter on increasing fractions of segmented documents (1 %, 2 %, 5 %, 10 %, …, 100 %) and plots F1 as a function of training-set size. Establishes the minimum-data claim.
**Inputs:** Outputs of notebook 01.
**Outputs:** `outputs/figure3_learning_curve.{png,pdf}`, `outputs/learning_curve_results.json`.
**Expected runtime (CPU baseline):** ~15 min on CPU (many short fits).

All randomness uses `SEED = 42`.

## Setup
This notebook uses shared utilities from `_setup.py`:
- `load_corpora()`: returns dict of dataframes per language
- `POS_HARMONIZATION`: dict mapping raw POS tags to unified tagset
- `BASE_PATH`: data directory (set this for your environment)

```python
from _setup import load_corpora, POS_HARMONIZATION, BASE_PATH
```


In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)
try:
    import torch
    torch.manual_seed(42)
except ImportError:
    pass

from _setup import load_corpora, POS_HARMONIZATION, BASE_PATH, set_seeds
set_seeds(42)


In [ ]:
# --- data-availability guard ------------------------------------------
missing = [l for l in ('akk', 'sux', 'elx') if l not in corpora]
if missing:
    print(f'WARNING: {missing} not loaded. Cells that depend on them will be skipped.')
    print(f'Set $CUNEI_DATA to a directory containing alltexts_AKK.csv, alltexts_SUX.csv, and the Elamite files.')
# Convenience: expose datasets dict for cells originally from the monolith.
datasets = {l: corpora[l] for l in ('akk', 'sux', 'elx') if l in corpora}
sign_dict = corpora.get('_sign_dict', {})


In [ ]:
corpora = load_corpora(BASE_PATH)
doc_corpora = corpora['_documents']


## Learning Curve Experimment

In [ ]:
# ============================================================
# LEARNING CURVE: TP vs MORFESSOR ON AKKADIAN
#
# Protocol:
#   1. Hold out 500 Akkadian docs as a fixed test set.
#   2. Shuffle remaining docs into a training pool with a fixed seed.
#   3. For each target token budget in {1K, 5K, 10K, 50K, 100K, 500K, ALL}:
#        a. Take the prefix of the training pool that reaches the budget.
#        b. Train TP and Morfessor on the prefix.
#        c. Evaluate both on the fixed held-out test set.
#   4. Plot F1 vs training corpus size.
#
# Matches the protocol of the rest of the paper:
#   - TP tunes threshold on test (same as Tables 2 and 3)
#   - Morfessor uses default corpusweight = 1.0
# ============================================================
import random
from collections import Counter
import numpy as np
import morfessor
from cunei_tools import CuneiSeg


# ----- 1. Split off a fixed held-out test set -----
random.seed(42)
docs_akk = [d for d in doc_corpora['akk']['unicode'].values()
            if d and len(d.split()) > 2]
random.shuffle(docs_akk)

N_TEST = 500
test_docs = docs_akk[:N_TEST]
train_pool = docs_akk[N_TEST:]

test_tokens = sum(len(d.split()) for d in test_docs)
pool_tokens = sum(len(d.split()) for d in train_pool)
print(f"Test set:      {len(test_docs):>6d} docs, {test_tokens:>10,d} tokens")
print(f"Training pool: {len(train_pool):>6d} docs, {pool_tokens:>10,d} tokens")

# Cumulative token counts let us slice the pool to any target size
cum_tokens = np.cumsum([len(d.split()) for d in train_pool])


# ----- 2. Helper functions -----
def get_train_subset(target_tokens):
    """Return the prefix of train_pool whose cumulative token count first
    reaches target_tokens."""
    idx = int(np.searchsorted(cum_tokens, target_tokens))
    idx = min(idx, len(train_pool) - 1)
    return train_pool[:idx + 1]


def train_morfessor(train_docs, corpusweight=1.0):
    wc = Counter()
    for doc in train_docs:
        for w in doc.split():
            if w:
                wc[w] += 1
    m = morfessor.BaselineModel(corpusweight=corpusweight)
    m.load_data([(c, w) for w, c in wc.items()])
    m.train_batch()
    return m


def _gold_boundaries(seg):
    g, pos = set(), 0
    for ch in seg:
        if ch == ' ':
            g.add(pos)
        else:
            pos += 1
    return g


def _predict_boundaries_morf(model, continuous):
    if not continuous:
        return set()
    try:
        segs, _ = model.viterbi_segment(continuous)
    except Exception:
        return set()
    b, cur = set(), 0
    for s in segs[:-1]:
        cur += len(s)
        b.add(cur)
    return b


def evaluate_morfessor(model, test_docs):
    tp, fp, fn = 0, 0, 0
    for d in test_docs:
        if not d or len(d.split()) < 2:
            continue
        gold = _gold_boundaries(d)
        pred = _predict_boundaries_morf(model, d.replace(' ', ''))
        tp += len(pred & gold)
        fp += len(pred - gold)
        fn += len(gold - pred)
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return {'f1': f1, 'precision': p, 'recall': r}


# ----- 3. Run the curve -----
TARGET_SIZES = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000, int(cum_tokens[-1])]
results = []

for size in TARGET_SIZES:
    train_docs = get_train_subset(size)
    actual_tokens = sum(len(d.split()) for d in train_docs)
    print(f"\n--- Target: {size:>10,d} | Actual: {actual_tokens:>10,d} tokens | "
          f"{len(train_docs):>5d} docs ---")

    # TP
    print("  TP training...", end='', flush=True)
    tp_seg = CuneiSeg(lang='akk')
    tp_seg.train(train_docs)
    tp_metrics = tp_seg.find_optimal_threshold(test_docs)
    print(f" F1 = {tp_metrics['f1']:.4f} (theta = {tp_metrics.get('theta', '?')})")

    # Morfessor
    print("  Morfessor training...", end='', flush=True)
    morf = train_morfessor(train_docs)
    morf_metrics = evaluate_morfessor(morf, test_docs)
    print(f" F1 = {morf_metrics['f1']:.4f}")

    results.append({
        'target_tokens': size,
        'actual_tokens': actual_tokens,
        'n_docs': len(train_docs),
        'tp_f1': float(tp_metrics['f1']),
        'tp_theta': float(tp_metrics.get('theta', 0)),
        'morf_f1': float(morf_metrics['f1']),
        'morf_precision': float(morf_metrics['precision']),
        'morf_recall': float(morf_metrics['recall']),
    })

print("\n=== Learning curve complete ===")
for r in results:
    print(f"  {r['actual_tokens']:>10,d} tokens | "
          f"TP F1 = {r['tp_f1']:.4f} | Morfessor F1 = {r['morf_f1']:.4f}")

In [ ]:
# ----- 4. Save results to JSON (locally and to Drive) -----
import json

out = {'protocol': {'n_test_docs': N_TEST, 'test_tokens': int(test_tokens),
                    'pool_tokens': int(pool_tokens), 'seed': 42,
                    'morfessor_corpusweight': 1.0},
       'results': results}

with open('/content/learning_curve_results.json', 'w') as f:
    json.dump(out, f, indent=2)
try:
    with open(BASE_PATH + 'learning_curve_results.json', 'w') as f:
        json.dump(out, f, indent=2)
    print(f"Saved /content/learning_curve_results.json")
    print(f"Saved {BASE_PATH}learning_curve_results.json")
except Exception as e:
    print(f"Local save only: {e}")

In [ ]:
# ----- 5. Generate Figure 3 -----
import matplotlib.pyplot as plt

sizes = [r['actual_tokens'] for r in results]
tp_f1s = [r['tp_f1'] for r in results]
morf_f1s = [r['morf_f1'] for r in results]

fig, ax = plt.subplots(figsize=(7, 4.5), dpi=140)
ax.plot(sizes, tp_f1s, 'o-', label='TP', color='#2c7fb8',
        linewidth=2, markersize=9, markeredgecolor='black', markeredgewidth=0.5)
ax.plot(sizes, morf_f1s, 's-', label='Morfessor', color='#e6802c',
        linewidth=2, markersize=9, markeredgecolor='black', markeredgewidth=0.5)

ax.set_xscale('log')
ax.set_xlabel('Training corpus size (tokens, log scale)', fontsize=11)
ax.set_ylabel('Held-out boundary F1', fontsize=11)
ax.set_ylim(-0.02, 1.05)
ax.set_yticks(np.arange(0, 1.05, 0.2))
ax.axhline(y=0.95, color='gray', linestyle='--', linewidth=0.6, alpha=0.5)
ax.text(sizes[-1]*0.95, 0.965, '0.95', color='gray', fontsize=9, ha='right')

ax.legend(loc='lower right', fontsize=11, frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.25, linestyle=':')

plt.tight_layout()
plt.savefig('/content/fig3_learning_curve.png', dpi=200, bbox_inches='tight')
plt.savefig('/content/fig3_learning_curve.pdf', bbox_inches='tight')
try:
    plt.savefig(BASE_PATH + 'fig3_learning_curve.png', dpi=200, bbox_inches='tight')
    plt.savefig(BASE_PATH + 'fig3_learning_curve.pdf', bbox_inches='tight')
    print(f"Saved to: {BASE_PATH}fig3_learning_curve.png and .pdf")
except Exception as e:
    print(f"Drive save failed: {e}")
    print("Saved to /content/ only")
plt.show()

# Print summary for the paper
print("\n=== SUMMARY FOR PAPER ===")
# Find where TP first reaches 0.95
for r in results:
    if r['tp_f1'] >= 0.95:
        print(f"TP reaches F1 ≥ 0.95 at {r['actual_tokens']:,} tokens")
        break
for r in results:
    if r['morf_f1'] >= 0.95:
        print(f"Morfessor reaches F1 ≥ 0.95 at {r['actual_tokens']:,} tokens")
        break
# Find crossover
for i in range(1, len(results)):
    if (results[i-1]['tp_f1'] > results[i-1]['morf_f1']) and \
       (results[i]['tp_f1'] <= results[i]['morf_f1']):
        print(f"Curves cross between {results[i-1]['actual_tokens']:,} and "
              f"{results[i]['actual_tokens']:,} tokens")
        break

In [ ]:
"""
EXPERIMENT 2 COMPLETE (ZERO-SHOT): Full 3x3 Transfer Matrix with 1000-Iteration Bootstrap CIs
Evaluates word boundary inference across all language pairs, fixing theta at
the SOURCE language's training optimum for a true zero-shot cross-lingual transfer.
"""
import random
from collections import Counter
import numpy as np

# Ensure strict reproducibility for bootstrap sampling
random.seed(42)
np.random.seed(42)

# Languages to evaluate (Assuming 'akk', 'sux', 'elx' based on cuneiform philology track)
languages = ['akk', 'sux', 'elx']

# ============================================================================
# ----- Step 1: Uniform Data Building -----
# ============================================================================
def build_world_a_docs(raw_corpora):
    out = {}
    items = raw_corpora.items() if isinstance(raw_corpora, dict) else [('doc_0', raw_corpora)]
    for doc_id, text_str in items:
        if not text_str or len(text_str.strip()) == 0:
            continue
        raw_words = text_str.strip().split(' ')
        doc_words = []
        for word in raw_words:
            glyph_list = [glyph for glyph in word if glyph.strip()]
            if glyph_list:
                doc_words.append(glyph_list)
        if len(doc_words) >= 2:
            out[doc_id] = doc_words
    return out

# Rebuild clean document maps for all languages
segmented_corpora = {}
for lang in languages:
    segmented_corpora[lang] = build_world_a_docs(doc_corpora[lang]['unicode'])

# Split each corpus into 80% train / 20% test up front to maintain true unseen test horizons
splits = {}
for lang, docs in segmented_corpora.items():
    ids = sorted(docs.keys())
    random.shuffle(ids)
    split_idx = int(len(ids) * 0.8)
    splits[lang] = {
        'train_ids': ids[:split_idx],
        'test_ids': ids[split_idx:]
    }

print("Data Split Complete:")
for lang in languages:
    print(f"  {lang.upper()}: Train={len(splits[lang]['train_ids'])} docs, Test={len(splits[lang]['test_ids'])} docs")
print("-" * 70)

# ============================================================================
# ----- Step 2: Core Mathematical Helpers -----
# ============================================================================
def compute_tp(train_ids, docs):
    bi, uni = Counter(), Counter()
    for did in train_ids:
        s = [sign for w in docs[did] for sign in w]
        for i in range(len(s)):
            uni[s[i]] += 1
            if i < len(s) - 1:
                bi[(s[i], s[i + 1])] += 1
    return {k: c / uni[k[0]] for k, c in bi.items() if uni[k[0]] > 0}

def evaluate_tp_on_docs(doc_list, tp, theta):
    tc = fc = fnc = 0
    for words in doc_list:
        s = [sign for w in words for sign in w]
        gold = set()
        pos = 0
        for w in words:
            pos += len(w)
            gold.add(pos)
        gold.discard(pos) # Remove training boundary trailing edge

        # Baseline fix: unseen bigrams default to 1.0 (highly cohesive)
        pred = {i + 1 for i in range(len(s) - 1)
                if tp.get((s[i], s[i + 1]), 0.0) < theta}

        tc += len(pred & gold)
        fc += len(pred - gold)
        fnc += len(gold - pred)

    p = tc / (tc + fc) if (tc + fc) else 0.0
    r = tc / (tc + fnc) if (tc + fnc) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return f1

def find_best_theta(doc_list, tp):
    best = {'f1': -1, 'theta': 0.5}
    for theta in [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
        f1 = evaluate_tp_on_docs(doc_list, tp, theta)
        if f1 > best['f1']:
            best = {'f1': f1, 'theta': theta}
    for theta in np.arange(max(0.01, best['theta'] - 0.12), min(0.99, best['theta'] + 0.12), 0.01):
        f1 = evaluate_tp_on_docs(doc_list, tp, theta)
        if f1 > best['f1']:
            best = {'f1': f1, 'theta': float(theta)}
    return best

# ============================================================================
# ----- Step 3: Matrix Generation & 1000-Iteration Bootstrap Loop -----
# ============================================================================
results_matrix = {src: {tgt: {} for tgt in languages} for src in languages}
n_bootstraps = 1000

print("Running 3x3 ZERO-SHOT Transfer Matrix and Bootstrap CIs...")
for src_lang in languages:
    # 1. Train the model on Source Train Corpus
    src_train_docs = [segmented_corpora[src_lang][tid] for tid in splits[src_lang]['train_ids']]
    tp_model = compute_tp(splits[src_lang]['train_ids'], segmented_corpora[src_lang])

    # ZERO-SHOT FIX: Optimize theta strictly within the source training landscape
    src_theta_opt = find_best_theta(src_train_docs, tp_model)['theta']
    print(f"\n[{src_lang.upper()} Trained Model] Source Theta Opt: {src_theta_opt:.2f}")

    for tgt_lang in languages:
        # Get target out-of-sample Test Set
        tgt_test_docs = [segmented_corpora[tgt_lang][tid] for tid in splits[tgt_lang]['test_ids']]
        n_docs = len(tgt_test_docs)

        # Empirical Point Estimate F1 forcing the Source language theta
        point_f1 = evaluate_tp_on_docs(tgt_test_docs, tp_model, src_theta_opt)

        # Bootstrap Resampling Loop (forcing fixed src_theta_opt)
        bootstrap_f1s = []
        for _ in range(n_bootstraps):
            # Sample with replacement from the target test documents
            boot_sample = random.choices(tgt_test_docs, k=n_docs)
            boot_f1 = evaluate_tp_on_docs(boot_sample, tp_model, src_theta_opt)
            bootstrap_f1s.append(boot_f1)

        # Calculate 95% Confidence Interval boundaries
        low_p = np.percentile(bootstrap_f1s, 2.5)
        high_p = np.percentile(bootstrap_f1s, 97.5)

        # Half-width delta calculation: (High - Low) / 2
        half_width = (high_p - low_p) / 2.0

        results_matrix[src_lang][tgt_lang] = {
            'f1': point_f1,
            'half_width': half_width,
            'theta': src_theta_opt
        }
        print(f"  -> Transfer to {tgt_lang.upper()}: Zero-Shot F1 = {point_f1:.4f} ± {half_width:.4f}")

# ============================================================================
# ----- Step 4: Format Outputs for LaTeX Verification -----
# ============================================================================
print("\n" + "=" * 70)
print("COMPRESSED ZERO-SHOT MATRIX FOR LATEX CHECK (vs paper_v4.tex)")
print("=" * 70)
print(f"{'Source -> Target':<18} | {'Point F1':<10} | {'CI Half-Width':<14} | {'Forced Theta'}")
print("-" * 70)

for src in languages:
    for tgt in languages:
        cell = results_matrix[src][tgt]
        label = f"{src.upper()} -> {tgt.upper()}"
        print(f"{label:<18} | {cell['f1']:<10.4f} | ± {cell['half_width']:<12.4f} | {cell['theta']:.2f}")

print("=" * 70)

In [ ]:
def visualize_predictions(doc_list, tp, theta, num_examples=3):
    print("=" * 70)
    print(f"VISUALIZATION TRACE (Threshold Theta = {theta:.2f})")
    print("=" * 70)

    for idx, words in enumerate(doc_list[:num_examples]):
        print(f"\n--- EXAMPLE {idx + 1} ---")

        # 1. Reconstruct gold visual string
        gold_str = " ".join(["".join(w) for w in words])
        print(f"Gold String: '{gold_str}'")

        # 2. Continuous sign stream
        s = [sign for w in words for sign in w]

        # 3. Step-by-step transition trace
        print("\nSign-by-Sign Evaluation Trace:")
        print(f"{'Transition':<12} | {'TP Score':<10} | {'Below Theta?':<12} | {'Action Taken'}")
        print("-" * 55)

        pred_segments = []
        current_word = [s[0]]

        for i in range(len(s) - 1):
            c1, c2 = s[i], s[i+1]
            score = tp.get((c1, c2), 1.0)
            below_thresh = score < theta

            action = "INSERT SPACE" if below_thresh else "GLUE TOGETHER"
            print(f"'{c1}' -> '{c2}'{'' : <5} | {score:<10.4f} | {str(below_thresh):<12} | {action}")

            if below_thresh:
                pred_segments.append("".join(current_word))
                current_word = [c2]
            else:
                current_word.append(c2)

        pred_segments.append("".join(current_word))
        pred_str = " ".join(pred_segments)

        # 4. Final Comparison
        print("-" * 55)
        print(f"Model Prediction: '{pred_str}'")
        print(f"Match Success?   {gold_str == pred_str}")

# Run the visualization on a few test documents
visualize_predictions(test_docs_list, tp_model, best_theta, num_examples=2)